<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</figure>

# Módulos propios en Python
**Taller de Programación para el Análisis de Datos — FCE-UBA**

> *Hasta ahora escribimos funciones dentro de las notebooks. El problema: si necesitamos la misma función en otra notebook, la copiamos y pegamos. Los módulos resuelven eso.*

---

**Contenido:**
1. ¿Qué es un módulo?
2. Estructura de un archivo `.py`
3. Las tres formas de importar
4. Funciones públicas y privadas
5. `__all__` — control de la interfaz pública
6. `%autoreload` — modificar el módulo sin reiniciar el kernel
7. Ejercicio integrador

**Parte 2 — Paquetes y submódulos**

8. De módulo a paquete
9. `__init__.py` — la puerta de entrada
10. Cómo importar submódulos
11. Imports absolutos vs. relativos
12. ¿Dónde busca Python? — `sys.path`
13. `__name__` en paquetes
14. Ejercicio integrador — Parte 2

---
## 1. ¿Qué es un módulo?

Un **módulo** es simplemente un archivo `.py` con funciones, clases o variables que podemos reutilizar en otros archivos.

```
proyecto/
│
├── analisis.py          ← el módulo (funciones reutilizables)
│
├── modulos_propios.ipynb  ← esta notebook (usa el módulo)
├── yelp_dataset.ipynb     ← otra notebook (también puede usarlo)
└── polygon_api.ipynb      ← otra más
```

La ventaja: escribís la función **una sola vez** y la importás donde la necesitás.
Si hay un bug, lo corregís en un solo lugar.

Esto aplica a cualquier librería que ya usaron:
`pandas`, `requests`, `matplotlib` — son módulos escritos por otras personas que importamos con `import`.

---
## 2. Estructura de un archivo `.py`

Abrí el archivo `analisis.py` que está en la misma carpeta que esta notebook.
Tiene esta estructura:

```python
"""
analisis.py          ← docstring del módulo: qué hace y qué contiene
===========
...
"""

__all__ = ['resumen', 'detectar_outliers', ...]   ← interfaz pública

import statistics                                  ← dependencias

# ════════════════════════════
# SECCIÓN 1 — Estadísticas
# ════════════════════════════

def resumen(datos):          ← función pública
    """Docstring..."""       ← siempre documentar
    ...

def _percentil(datos, p):    ← función privada (empieza con _)
    ...
```

Las convenciones importantes:
- **Docstring del módulo** al principio: qué hace, qué contiene
- **`__all__`**: lista de lo que se exporta (lo vemos en la sección 5)
- **Funciones privadas** con `_` al inicio: son helpers internos, no parte de la interfaz pública

---
## 3. Las tres formas de importar

Python ofrece tres sintaxis para importar. Cada una tiene sus casos de uso.

In [1]:
# ── Forma 1: importar el módulo completo 
# Se accede a cada función como  modulo.funcion()
# Ventaja: siempre queda claro de dónde viene cada función

import analisis

datos = [15, 22, 18, 31, 19, 24, 17, 200, 21, 16]

resultado = analisis.resumen(datos)
print(resultado)

{'n': 10, 'media': 38.3, 'mediana': 20.0, 'desvio': 57.00691965795809, 'minimo': 15, 'maximo': 200, 'p25': 17.25, 'p75': 23.5}


In [ ]:
# ── Forma 2: importar funciones específicas
# Se llaman directamente, sin prefijo
# Ventaja: código más corto
# Desventaja: si hay dos funciones con el mismo nombre en distintos módulos,
#             la segunda sobreescribe a la primera

from analisis import detectar_outliers, formato_moneda

outliers = detectar_outliers(datos)
print("Outliers:", outliers)

print(formato_moneda(1234567.89))

In [ ]:
# ── Forma 3: importar con alias ───────────────────────────────────
# Útil cuando el nombre del módulo es largo o hay conflicto de nombres
# (pandas se importa siempre como pd, numpy como np — son convenciones)

import analisis as an

print(an.formato_porcentaje(0.1534))
print(an.formato_miles(7_000_000))

**¿Cuál usar?**

| Situación | Forma recomendada |
|-----------|-------------------|
| Módulo propio, pocas funciones | `from analisis import resumen` |
| Módulo propio, muchas funciones | `import analisis` |
| Librería estándar con convención (`pd`, `np`, `plt`) | `import pandas as pd` |
| Evitar ambigüedad entre dos módulos con funciones iguales | `import modulo_a`, `import modulo_b` |

---
## 4. Funciones públicas y privadas

En Python, las funciones que empiezan con `_` son **privadas por convención** — son funciones internas que no forman parte de la interfaz que le ofrecemos al usuario del módulo.

In [2]:
# Las funciones privadas son accesibles (Python no las bloquea),
# pero la convención dice que no deberían usarse desde afuera

# Uso correcto — función pública
print(analisis.resumen([1, 2, 3, 4, 5]))

# Posible pero no recomendado — función privada
# analisis._percentil([1, 2, 3, 4, 5], 25)

# Ver qué funciones tiene el módulo
publicas = [nombre for nombre in dir(analisis) if not nombre.startswith('_')]
print("\nFunciones públicas del módulo:", publicas)

{'n': 5, 'media': 3, 'mediana': 3, 'desvio': 1.5811388300841898, 'minimo': 1, 'maximo': 5, 'p25': 2.0, 'p75': 4.0}

Funciones públicas del módulo: ['detectar_outliers', 'formato_miles', 'formato_moneda', 'formato_porcentaje', 'normalizar', 'resumen', 'statistics']


---
## 5. `__all__` — control de la interfaz pública

`__all__` es una lista que define exactamente qué se exporta cuando alguien hace `from analisis import *`.

También sirve como documentación: al abrir el archivo `.py`, `__all__` al principio dice inmediatamente qué funciones son parte de la interfaz.

In [3]:
# Ver el __all__ del módulo
print("Interfaz pública definida en __all__:")
for nombre in analisis.__all__:
    print(f"  - {nombre}")

print()

# El docstring del módulo
print("Docstring del módulo:")
print(analisis.__doc__)

Interfaz pública definida en __all__:
  - resumen
  - detectar_outliers
  - normalizar
  - formato_moneda
  - formato_porcentaje
  - formato_miles

Docstring del módulo:

analisis.py
Módulo de utilidades para análisis y formateo de datos numéricos.

Secciones
---------
- Estadísticas : resumen, outliers, normalización
- Formateo     : moneda, porcentaje, miles



In [ ]:
# help() muestra el docstring de una función
help(analisis.detectar_outliers)

---
## 6. `%autoreload` — modificar el módulo sin reiniciar el kernel

Cuando importás un módulo, Python lo carga en memoria y **no lo vuelve a leer** aunque lo modifiques.
Si cambiás `analisis.py`, necesitarías reiniciar el kernel para ver el cambio.

`%autoreload` es una extensión de Jupyter que recarga automáticamente los módulos modificados antes de cada celda.

In [ ]:
# Activar autoreload — ejecutar esto al principio del notebook
# cuando estés desarrollando un módulo

%load_ext autoreload
%autoreload 2   #El número 2 significa que se recargarán todos los módulos antes de ejecutar cualquier sentencia de código (excepto aquellos que tú decidas excluir explícitamente).

# A partir de acá: cada vez que ejecutes una celda,
# Jupyter verifica si analisis.py cambió y lo recarga automáticamente.
# Podés editar el .py, guardar, y ejecutar la celda — sin reiniciar.

print("autoreload activado.")

autoreload activado.


Si no usás Jupyter o querés recargar manualmente, también existe `importlib.reload()`:

In [ ]:
import importlib

# Recargar el módulo manualmente
importlib.reload(analisis)

print("Módulo recargado.")

---
## Recorrido por las funciones del módulo

Veamos cada función con ejemplos concretos.

In [5]:
import analisis
from pprint import pprint   # pretty print para dicts

# ── resumen() ────────────────────────────────────────────────────
ventas = [120, 135, 98, 150, 160, 89, 143, 127, 500, 115, 132, 141]

print("Ventas mensuales:", ventas)
print()
pprint(analisis.resumen(ventas))

Ventas mensuales: [120, 135, 98, 150, 160, 89, 143, 127, 500, 115, 132, 141]

{'desvio': 109.2743281282424,
 'maximo': 500,
 'media': 159.16666666666666,
 'mediana': 133.5,
 'minimo': 89,
 'n': 12,
 'p25': 118.75,
 'p75': 144.75}


In [6]:
# ── detectar_outliers() ──────────────────────────────────────────
resultado = analisis.detectar_outliers(ventas)

print(f"Límite inferior: {resultado['limite_inf']:.1f}")
print(f"Límite superior: {resultado['limite_sup']:.1f}")
print(f"Outliers:        {resultado['outliers']}")

Límite inferior: 79.8
Límite superior: 183.8
Outliers:        [500]


In [7]:
# ── normalizar() ─────────────────────────────────────────────────
datos_brutos = [200, 450, 100, 800, 350]
normalizados = analisis.normalizar(datos_brutos)

print("Original:    ", datos_brutos)
print("Normalizado: ", [round(x, 3) for x in normalizados])
print(f"Rango: [{min(normalizados):.1f}, {max(normalizados):.1f}]")

Original:     [200, 450, 100, 800, 350]
Normalizado:  [0.143, 0.5, 0.0, 1.0, 0.357]
Rango: [0.0, 1.0]


In [8]:
# ── funciones de formateo ────────────────────────────────────────
print(analisis.formato_moneda(1_234_567.89))
print(analisis.formato_moneda(1_234_567.89, simbolo='USD'))
print(analisis.formato_moneda(1_500, decimales=0))
print()
print(analisis.formato_porcentaje(0.1534))    # proporción → porcentaje
print(analisis.formato_porcentaje(15.34))     # ya es porcentaje
print(analisis.formato_porcentaje(-0.05))
print()
print(analisis.formato_miles(7_000_000))
print(analisis.formato_miles(1_234_567))

$ 1.234.567,89
USD 1.234.567,89
$ 1.500

+15,3%
+15,3%
-5,0%

7.000.000
1.234.567


In [9]:
# ── Uso combinado: resumen formateado ────────────────────────────
ingresos = [85000, 92000, 78000, 105000, 88000, 91000, 250000, 83000, 96000, 87000]

stats  = analisis.resumen(ingresos)
atip   = analisis.detectar_outliers(ingresos)

print("=== Resumen de ingresos ===")
print(f"  N:       {analisis.formato_miles(stats['n'])}")
print(f"  Media:   {analisis.formato_moneda(stats['media'], decimales=0)}")
print(f"  Mediana: {analisis.formato_moneda(stats['mediana'], decimales=0)}")
print(f"  Desvío:  {analisis.formato_moneda(stats['desvio'], decimales=0)}")
print(f"  Mín:     {analisis.formato_moneda(stats['minimo'], decimales=0)}")
print(f"  Máx:     {analisis.formato_moneda(stats['maximo'], decimales=0)}")

if atip['outliers']:
    vals = ', '.join(analisis.formato_moneda(v, decimales=0) for v in atip['outliers'])
    print(f"\n⚠️  Outliers detectados: {vals}")

=== Resumen de ingresos ===
  N:       10
  Media:   $ 105.500
  Mediana: $ 89.500
  Desvío:  $ 51.310
  Mín:     $ 78.000
  Máx:     $ 250.000

⚠️  Outliers detectados: $ 250.000


---
## 7. Ejercicio integrador

Abrí `analisis.py` y agregá la siguiente función en la sección de Estadísticas:

```python
def coeficiente_variacion(datos):
    """
    Calcula el coeficiente de variación (CV): desvío / media.

    Indica qué tan dispersos están los datos en relación a la media.
    Un CV bajo significa datos homogéneos; un CV alto, mucha variabilidad.

    Retorna
    -------
    float — el CV como proporción (ej. 0.15 significa 15% de variabilidad)
    """
    # completar
    pass
```

Pasos:
1. Implementá la función en `analisis.py` y guardá el archivo
2. Ejecutá la celda de abajo — `%autoreload` recargará el módulo automáticamente
3. Verificá que el resultado sea correcto: para `[10, 10, 10]` el CV debería ser `0.0`

In [ ]:
# Probar la función una vez implementada
datos_a = [10, 10, 10, 10]          # sin variabilidad → CV = 0.0
datos_b = [10, 20, 30, 40, 50]      # alta variabilidad
datos_c = [100, 102, 99, 101, 100]  # muy baja variabilidad

for nombre, d in [('datos_a', datos_a), ('datos_b', datos_b), ('datos_c', datos_c)]:
    cv = analisis.coeficiente_variacion(d)
    print(f"{nombre}: CV = {analisis.formato_porcentaje(cv)}")

---
---
# Parte 2 — Paquetes y submódulos

> *`analisis.py` funciona bien, pero imaginemos que crece: 40 funciones de estadística, 15 de formateo, otras para leer archivos y otras para hacer reportes. Un solo archivo de 2000 líneas se vuelve inmanejable. La solución es la misma que usamos con los archivos de la compu: **carpetas**.*

---
## 8. De módulo a paquete

Un **paquete** es una carpeta que contiene módulos (`.py`) y un archivo especial `__init__.py`.
Cada `.py` dentro del paquete es un **submódulo**. Y una carpeta dentro del paquete (con su propio `__init__.py`) es un **subpaquete**.

Partimos `analisis.py` así:

```
proyecto/
│
├── analisis.py                 ← ANTES: todo en un archivo
│
├── analitica/                  ← DESPUÉS: un paquete
│   ├── __init__.py             ← convierte la carpeta en paquete
│   ├── __main__.py             ← se ejecuta con  python -m analitica
│   ├── estadisticas.py         ← submódulo: resumen, detectar_outliers, normalizar
│   ├── formateo.py             ← submódulo: formato_moneda, formato_porcentaje, formato_miles
│   ├── _utils.py               ← submódulo privado: percentil, separar_miles
│   │
│   └── reportes/               ← subpaquete
│       ├── __init__.py
│       └── consola.py          ← submódulo: imprimir_resumen
│
├── modulos_propios.ipynb
└── main_paquete.py
```

| Término | Qué es | Ejemplo | Se importa como |
|---|---|---|---|
| **Módulo** | un archivo `.py` | `analisis.py` | `import analisis` |
| **Paquete** | una carpeta con `__init__.py` | `analitica/` | `import analitica` |
| **Submódulo** | un `.py` dentro de un paquete | `analitica/formateo.py` | `import analitica.formateo` |
| **Subpaquete** | una carpeta con `__init__.py` dentro de un paquete | `analitica/reportes/` | `import analitica.reportes` |

El **punto** en `analitica.formateo` es el separador de carpetas: `analitica/formateo.py` → `analitica.formateo`.

Ya usaron muchos paquetes sin saberlo:

| Lo que escribieron | Paquete | Submódulo / subpaquete |
|---|---|---|
| `import matplotlib.pyplot as plt` | `matplotlib` | `pyplot` |
| `from sklearn.model_selection import train_test_split` | `sklearn` | `model_selection` |
| `os.path.exists(...)` | `os` | `path` |
| `pd.api.types.is_numeric_dtype(...)` | `pandas` | `api` → `types` |

---
## 9. `__init__.py` — la puerta de entrada del paquete

`__init__.py` se ejecuta **una sola vez**, la primera vez que se importa el paquete (o cualquier cosa de adentro).
Puede estar vacío, pero normalmente se usa para:

1. Documentar el paquete (docstring)
2. **Re-exportar** las funciones más usadas, para acortar los imports
3. Definir `__all__` y metadatos como `__version__`

Nuestro `analitica/__init__.py`:

```python
from .estadisticas import resumen, detectar_outliers, normalizar
from .formateo import formato_moneda, formato_porcentaje, formato_miles

__all__ = ['resumen', 'detectar_outliers', ...]
__version__ = '0.1.0'

print(f"[analitica] paquete inicializado (v{__version__})")   # solo para la clase
```

> El `print` está para ver cuándo se ejecuta. En un paquete real no se pone.

In [1]:
# Primera importación: se ejecuta __init__.py (aparece el print)
import analitica

print("Versión:", analitica.__version__)

[analitica] paquete inicializado (v0.1.0)
Versión: 0.1.0


In [2]:
# Segunda importación: Python ya lo tiene en memoria → NO se vuelve a ejecutar
import analitica
import analitica.formateo

# Python guarda todo lo importado en sys.modules
import sys
print([nombre for nombre in sys.modules if nombre.startswith('analitica')])

['analitica._utils', 'analitica.estadisticas', 'analitica.formateo', 'analitica']


Observen que al importar `analitica` se cargaron también `analitica.estadisticas`, `analitica.formateo` y `analitica._utils`: el `__init__.py` los importó para re-exportar sus funciones.

---
## 10. Cómo importar submódulos

Las tres formas de importar de la sección 3 siguen valiendo; ahora el "camino" tiene puntos.

In [3]:
# ── Forma 1: importar el submódulo con su ruta completa
import analitica.estadisticas

analitica.estadisticas.resumen([10, 20, 30])

{'n': 3,
 'media': 20,
 'mediana': 20,
 'desvio': 10.0,
 'minimo': 10,
 'maximo': 30,
 'p25': 15.0,
 'p75': 25.0}

In [4]:
# ── Forma 2: importar el submódulo desde el paquete (la más común)
from analitica import formateo

print(formateo.formato_moneda(1_234_567.89))
print(formateo.formato_porcentaje(0.1534))

$ 1.234.567,89
+15,3%


In [5]:
# ── Forma 3: alias para un submódulo (como  matplotlib.pyplot as plt)
import analitica.estadisticas as est

est.detectar_outliers([10, 12, 11, 13, 9, 100, 11])

{'outliers': [100], 'limite_inf': 7.5, 'limite_sup': 15.5}

In [6]:
# ── Forma 4: importar una función de un subpaquete
from analitica.reportes import imprimir_resumen

imprimir_resumen("Ingresos", [85000, 92000, 78000, 105000, 88000, 91000, 250000, 83000, 96000, 87000])


════════════════════════════════════════
  Ingresos
════════════════════════════════════════
  N        10
  Media    $ 105.500
  Mediana  $ 89.500
  Desvío   $ 51.310
  Mín      $ 78.000
  Máx      $ 250.000
  P25      $ 85.500
  P75      $ 95.000

  ⚠  Outliers: $ 250.000
     Rango normal: $ 71.250 — $ 109.250


In [ ]:
# No hace falta saber en qué submódulo está cada función:
from analitica import resumen, formato_miles

print(formato_miles(resumen(range(1_000_000))['n']))

# Es literalmente el mismo objeto función, accesible por dos caminos
print(analitica.resumen is analitica.estadisticas.resumen)

1.000.000
True


**¿Cuál usar?**

| Situación | Forma recomendada |
|---|---|
| Función muy usada que el paquete re-exporta | `from analitica import resumen` |
| Varias funciones de un mismo submódulo | `from analitica import formateo` → `formateo.formato_moneda(...)` |
| Submódulo con nombre largo o convención | `import matplotlib.pyplot as plt` |
| Algo específico de un subpaquete | `from analitica.reportes import imprimir_resumen` |

---
## 11. Imports absolutos vs. relativos

**Adentro** del paquete, los submódulos se importan entre sí. Hay dos maneras:

| Tipo | Ejemplo (dentro de `reportes/consola.py`) | Significado |
|---|---|---|
| Absoluto | `from analitica.estadisticas import resumen` | ruta completa desde el paquete raíz |
| Relativo | `from ..estadisticas import resumen` | `..` = subir un nivel (a `analitica`) |
| Relativo | `from ._utils import percentil` (en `estadisticas.py`) | `.` = este mismo paquete |

Es como `cd .` y `cd ..` en la terminal.

Los relativos tienen una ventaja: si mañana renombran el paquete `analitica` → `mi_libreria`, los imports internos no cambian.

⚠️ **Los imports relativos solo funcionan dentro de un paquete.** Si ejecutan un submódulo como archivo suelto, Python no sabe a qué paquete pertenece:

In [9]:
!python -m analitica

[analitica] paquete inicializado (v0.1.0)

════════════════════════════════════════
  Demo del paquete analitica
════════════════════════════════════════
  N        7
  Media    $ 179
  Mediana  $ 135
  Desvío   $ 144
  Mín      $ 89
  Máx      $ 500
  P25      $ 109
  P75      $ 155

  ⚠  Outliers: $ 500
     Rango normal: $ 40 — $ 224


---
## 12. ¿Dónde busca Python los módulos? — `sys.path`

Cuando escriben `import analitica`, Python recorre en orden una lista de carpetas hasta encontrar `analitica.py` o una carpeta `analitica/`.
Esa lista es `sys.path`. Incluye la **carpeta actual** (en un script es la carpeta del `.py`; en Jupyter aparece como `''`), por eso funciona sin instalar nada. Después vienen la librería estándar y `site-packages`, donde `pip` instala `pandas`, `requests`, etc.

In [ ]:
import sys

for ruta in sys.path:
    print(repr(ruta))

print("\nUbicación del paquete:", analitica.__file__)
print("Ubicación del submódulo:", analitica.formateo.__file__)

Si el paquete está en **otra carpeta** (por ejemplo, una carpeta común para todas las materias), se puede agregar a `sys.path`:

```python
import sys
sys.path.append(r"C:\Users\yo\mis_librerias")   # Windows
sys.path.append("/home/yo/mis_librerias")        # Linux / Mac

import analitica   # ahora lo encuentra
```

⚠️ **Cuidado con los nombres.** Si crean un archivo `statistics.py` o `csv.py` en su carpeta, Python va a encontrar **ese** antes que el de la librería estándar (porque su carpeta está primera en `sys.path`), y los imports de la librería se rompen. Por eso nuestro paquete se llama `analitica` y no `statistics`.

> La forma "profesional" de no depender de `sys.path` es convertir el paquete en instalable (`pyproject.toml` + `pip install -e .`). Es el paso siguiente para publicar una librería, pero excede esta clase.

---
## 13. `__name__` en paquetes

En la parte 1 vimos `if __name__ == "__main__":`. Dentro de un paquete, `__name__` de cada submódulo es su **ruta completa con puntos**:

In [11]:
print(analitica.__name__)
print(analitica.estadisticas.__name__)
print(analitica.reportes.consola.__name__)

analitica
analitica.estadisticas
analitica.reportes.consola


Por eso:
- `python -m analitica.reportes.consola` → dentro del archivo, `__name__ == "__main__"`
- `import analitica.reportes.consola` → `__name__ == "analitica.reportes.consola"`

Y el archivo `__main__.py` es el "punto de entrada" del paquete cuando se ejecuta con `python -m analitica`.

---
## Antes y después

`main_simple.py` (con el módulo único) tenía 25 líneas de `print` para mostrar el resumen, y `main_csv.py` repetía esas mismas líneas.
Ahora esa lógica vive en un solo lugar — `analitica/reportes/consola.py` — y `main_paquete.py` queda así:

```python
from analitica.reportes import imprimir_resumen

ventas = [120, 135, 98, 150, 160, 89, 143, 127, 500, 115, 132, 141]

if __name__ == "__main__":
    imprimir_resumen("Ventas mensuales", ventas)
```

In [12]:
!python main_paquete.py

[analitica] paquete inicializado (v0.1.0)

════════════════════════════════════════
  Ventas mensuales
════════════════════════════════════════
  N        12
  Media    $ 159
  Mediana  $ 134
  Desvío   $ 109
  Mín      $ 89
  Máx      $ 500
  P25      $ 119
  P75      $ 145

  ⚠  Outliers: $ 500
     Rango normal: $ 80 — $ 184
